# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ziadhamouda370-beep/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State itOne row represents one pseudonymized content item (page). The starter dataset contains aggregated search and engagement metrics over a trailing 90-day window ending at export time. I will treat the content item as the unit of analysis and use the same 90-day window consistently for this starter-data contract., then verify it below.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/ziadhamouda370-beep/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

Rows: 30000
Columns: 44
Unique content IDs: 30000
Duplicate content IDs: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*Features: search and engagement signals available before modeling, including search_volume, competition, cpc, word_count, content_age_days, days_since_last_update, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions, impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d, ctr, avg_position, and engagement_rate. Label: trend_direction, where down defines the declining class. Context: content_id, client_id, content_type, main_intent, and tier fields for grouping and interpretation. Excluded: trend_pct and trend_direction are excluded from features because they define or directly encode the label; content_id and client_id are excluded because they are identifiers, not predictive signals. provider_used and model_used are also excluded because the data dictionary marks them as not model features.**

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_candidates = [
    "search_volume", "competition", "cpc",
    "word_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "ctr", "avg_position", "engagement_rate"
]

label = "trend_direction"

excluded = [
    "trend_pct", "content_id", "client_id",
    "provider_used", "model_used"
]

print("Feature candidates:", len(feature_candidates))
print("Label:", label)
print("Excluded:", excluded)
print("Missing feature columns:", [c for c in feature_candidates if c not in df.columns])

Feature candidates: 25
Label: trend_direction
Excluded: ['trend_pct', 'content_id', 'client_id', 'provider_used', 'model_used']
Missing feature columns: []


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*I will verify the contract directly from the dataframe: row count, unique content IDs, duplicate IDs, target distribution, missing values in key fields, and the available 30-day and 90-day activity columns. These checks confirm that the stated grain and fields are actually present rather than assumed.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nUnique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

print("\nTarget distribution:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nMissing values in selected fields:")
check_cols = [
    "search_volume",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "trend_direction"
]

print(df[check_cols].isna().sum())

print("\nWindow fields present:")
window_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print([c for c in window_cols if c in df.columns])

Rows: 30000
Columns: 44

Unique content IDs: 30000
Duplicate content IDs: 0

Target distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Missing values in selected fields:
search_volume           2468
word_count              7699
impressions_90d            0
clicks_90d                 0
sessions_90d               0
impressions_last_30d       0
impressions_prev_30d       0
trend_direction            0
dtype: int64

Window fields present:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*The data has important limits. The starter dataset is an aggregated snapshot rather than a complete time series, so it cannot establish long-term historical behavior. Some early rows may have GSC-only coverage, which can make comparisons uneven. The 30-day and 90-day windows can also overlap, so they should not be treated as independent observations. The dataset can support observed and directional analysis, but it cannot prove causality or predict Google's ranking algorithm.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Data limits check")

print("Rows:", len(df))

print("\n90-day fields:")
print([
    c for c in df.columns
    if "90d" in c
])

print("\n30-day fields:")
print([
    c for c in df.columns
    if "30d" in c
])

print("\nPotential label leakage fields:")
print([
    c for c in ["trend_direction", "trend_pct"]
    if c in df.columns
])

Data limits check
Rows: 30000

90-day fields:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d']

30-day fields:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Potential label leakage fields:
['trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.